**Name** Yasser Khan

**Z-Number** Z23971583

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import random

RANDOM_SEED     = 42
FROM_COLUMN     = 'from'
TO_COLUMN       = 'to'
TARGET_COLUMN   = 'link'
FEATURE_COLUMNS = ['common_neighbors', 'jaccard_coefficient',
                   'adamic_adar', 'preferential_attachment']

# Load graph
df_edges = pd.read_csv('/content/musae_ENGB_edges.csv')
G = nx.Graph()
G.add_edges_from(zip(df_edges[FROM_COLUMN], df_edges[TO_COLUMN]))

number_of_edges = G.number_of_edges()
print(f"{'Nodes:':<35} {G.number_of_nodes():>10,}")
print(f"{'Edges:':<35} {number_of_edges:>10,}")

In [ ]:
# Sample negatives — same logic as PySpark notebook
random.seed(RANDOM_SEED)
non_edges        = list(nx.non_edges(G))
negative_samples = random.sample(non_edges, number_of_edges)

samples = [(u, v, 1) for u, v in G.edges()] + \
          [(u, v, 0) for u, v in negative_samples]

random.seed(RANDOM_SEED)
random.shuffle(samples)
print(f'Total samples: {len(samples):,}')

In [ ]:
# Compute features (takes a few minutes)
node_pairs   = [(u, v) for u, v, _ in samples]

jaccard_dict = {(u, v): s for u, v, s in nx.jaccard_coefficient(G,     node_pairs)}
adamic_dict  = {(u, v): s for u, v, s in nx.adamic_adar_index(G,       node_pairs)}
pa_dict      = {(u, v): s for u, v, s in nx.preferential_attachment(G, node_pairs)}

rows = []
for u, v, label in samples:
    cn = len(list(nx.common_neighbors(G, u, v)))
    rows.append({
        FROM_COLUMN:               u,
        TO_COLUMN:                 v,
        'common_neighbors':        cn,
        'jaccard_coefficient':     jaccard_dict[(u, v)],
        'adamic_adar':             adamic_dict[(u, v)],
        'preferential_attachment': pa_dict[(u, v)],
        TARGET_COLUMN:             label,
    })

df = pd.DataFrame(rows)
print('Features computed.')
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df[FEATURE_COLUMNS].values.astype(float)
y = df[TARGET_COLUMN].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print(f'Training set: {len(X_train):,}')
print(f'Test set:     {len(X_test):,}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)),
])

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = {'AUC': [], 'Accuracy': [], 'Weighted Precision': [],
             'Weighted Recall': [], 'Weighted F1': []}

for train_idx, val_idx in kf.split(X_train, y_train):
    Xtr, Xval = X_train[train_idx], X_train[val_idx]
    ytr, yval = y_train[train_idx], y_train[val_idx]

    pipeline.fit(Xtr, ytr)
    y_pred = pipeline.predict(Xval)
    y_prob = pipeline.predict_proba(Xval)[:, 1]

    cv_scores['AUC'].append(roc_auc_score(yval, y_prob))
    cv_scores['Accuracy'].append(accuracy_score(yval, y_pred))
    cv_scores['Weighted Precision'].append(precision_score(yval, y_pred, average='weighted', zero_division=0))
    cv_scores['Weighted Recall'].append(recall_score(yval, y_pred, average='weighted'))
    cv_scores['Weighted F1'].append(f1_score(yval, y_pred, average='weighted'))

print('Average CV Metrics:')
for name, scores in cv_scores.items():
    print(f'  {name:<25}: {np.mean(scores):.2f}')

In [ ]:
from sklearn.metrics import classification_report

pipeline.fit(X_train, y_train)
y_pred_test = pipeline.predict(X_test)
y_prob_test = pipeline.predict_proba(X_test)[:, 1]

print(f'AUC:      {roc_auc_score(y_test, y_prob_test):.2f}')
print(f'Accuracy: {accuracy_score(y_test, y_pred_test):.2f}')
print()
print(classification_report(y_test, y_pred_test, target_names=['Non-Link', 'Link']))

In [ ]:
print('Model Coefficients:')
lr_model = pipeline.named_steps['lr']
for name, coef in zip(FEATURE_COLUMNS, lr_model.coef_[0]):
    print(f'  {name:<30}: {coef:.3f}')